<a href="https://colab.research.google.com/github/tol/osint/blob/main/notebooks/Pnnr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Install  depdendencies

In [ ]:
! pip install pandas requests openpyxl

### Fetch all pnnr  payments

In [ ]:
#!/usr/bin/env python3
"""
Script to fetch all PNRR payment records from the API
and save them to an Excel file.

Usage: python fetch_all_pnrr.py
"""

import requests
import json
import pandas as pd
import time
from datetime import datetime

def fetch_all_records():
    """Fetch all records from the PNRR API."""

    base_url = "https://pnrr.fonduri-ue.ro/ords/pnrr/mfe/plati_pnrr"
    all_items = []
    offset = 0
    limit = 1000
    has_more = True
    batch_count = 0

    print("=" * 70)
    print("PNRR Payment Records - Complete Data Fetcher")
    print("=" * 70)
    print(f"Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("-" * 70)

    while has_more:
        try:
            url = f"{base_url}?limit={limit}&offset={offset}"

            print(f"Fetching batch {batch_count + 1} (offset: {offset})...", end=" ")

            response = requests.get(url, timeout=30,verify=False)
            response.raise_for_status()
            data = response.json()

            # Add items from this batch
            items = data.get('items', [])
            all_items.extend(items)

            batch_count += 1
            has_more = data.get('hasMore', False)
            current_count = len(items)
            total_so_far = len(all_items)

            print(f"âœ“ {current_count} records (Total: {total_so_far:,})")

            # Update offset for next request
            offset += limit

            # Small delay to be respectful to the API
            if has_more:
                time.sleep(0.5)

        except requests.exceptions.RequestException as e:
            print(f"\nâœ— Error on batch {batch_count + 1}: {e}")
            print(f"Successfully retrieved {len(all_items):,} records before error.")
            break
        except Exception as e:
            print(f"\nâœ— Unexpected error: {e}")
            break

    print("-" * 70)
    print(f"âœ“ Fetching complete!")
    print(f"âœ“ Total records retrieved: {len(all_items):,}")
    print(f"Finished at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    return all_items

def save_to_files(items):
    """Save data to both JSON and Excel formats."""

    if not items:
        print("No data to save!")
        return

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    # Save to JSON
    json_filename = f'pnrr_all_records_{timestamp}.json'
    print(f"\nSaving to JSON: {json_filename}...", end=" ")
    with open(json_filename, 'w', encoding='utf-8') as f:
        json.dump(items, f, ensure_ascii=False, indent=2)
    print("âœ“")

    # Convert to DataFrame
    df = pd.DataFrame(items)

    # Save to Excel
    excel_filename = f'PNRR_All_Records_{timestamp}.xlsx'
    print(f"Saving to Excel: {excel_filename}...", end=" ")

    with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='Plati PNRR', index=False)

        # Auto-adjust column widths
        worksheet = writer.sheets['Plati PNRR']
        for column in worksheet.columns:
            max_length = 0
            column_letter = column[0].column_letter
            for cell in column:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
            adjusted_width = min(max_length + 2, 50)
            worksheet.column_dimensions[column_letter].width = adjusted_width

    print("âœ“")

    # Display summary statistics
    print("\n" + "=" * 70)
    print("DATA SUMMARY")
    print("=" * 70)
    print(f"Total records: {len(df):,}")
    print(f"Total columns: {len(df.columns)}")
    print(f"\nColumns:")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i:2d}. {col}")

    # Financial summary
    if 'valoare_plata_fe' in df.columns:
        total_ron = df['valoare_plata_fe'].sum()
        print(f"\nTotal payments (RON): {total_ron:,.2f}")

    if 'valoare_plata_fe_euro' in df.columns:
        total_euro = df['valoare_plata_fe_euro'].sum()
        print(f"Total payments (EUR): {total_euro:,.2f}")

    print("\n" + "=" * 70)
    print(f"âœ“ Files saved successfully!")
    print(f"  - JSON: {json_filename}")
    print(f"  - Excel: {excel_filename}")
    print("=" * 70)

def main():
    """Main function."""
    try:
        # Fetch all records
        items = fetch_all_records()

        # Save to files
        if items:
            save_to_files(items)
        else:
            print("\nâš ï¸ No records were fetched. Please check your internet connection.")
            print("   and verify the API is accessible.")

    except KeyboardInterrupt:
        print("\n\nâš ï¸ Process interrupted by user.")
    except Exception as e:
        print(f"\nâœ— Fatal error: {e}")

if __name__ == "__main__":
    main()

### Fetch all pnnr data

In [ ]:
#!/usr/bin/env python3
"""
Fetch ALL PNRR data: Plati + Angajamente + Persons + Measures
"""
import requests
import json
import pandas as pd
import time
from datetime import datetime

endpoints = {
    'plati': 'https://pnrr.fonduri-ue.ro/ords/pnrr/mfe/plati_pnrr',
    'angajamente': 'https://pnrr.fonduri-ue.ro/ords/pnrr/mfe/angajamente',
    'persons': 'https://pnrr.fonduri-ue.ro/ords/pnrr/mfe/persons',
    'measures': 'https://pnrr.fonduri-ue.ro/ords/pnrr/mfe/measures'
}

def fetch_all(base_url):
    all_items = []
    offset = 0
    limit = 1000

    while True:
        url = f"{base_url}?limit={limit}&offset={offset}"
        print(f"Fetching {url}...")

        resp = requests.get(url,verify=False)
        data = resp.json()
        items = data.get('items', [])

        if not items:
            break

        all_items.extend(items)
        offset += limit

        if not data.get('hasMore', False):
            break

        time.sleep(0.5)

    return all_items

# RUN IT
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
for name, url in endpoints.items():
    print("f=== {name.upper()} ===")
    items = fetch_all(url)

    if items:
        df = pd.DataFrame(items)
        filename = f'PNRR_{name}_{timestamp}.xlsx'
        df.to_excel(filename, index=False)
        print(f"✓ Saved {len(df)} records to {filename}")